Financial Analytics & Real Estate Market Intelligence  

Real estate companies serve a highly diverse buyer base: individual home buyers, first-time buyers, corporate and institutional investors, international buyers, and high-net-worth investors. When all of these are treated the same, the result is inefficient marketing spend, generic property recommendations, and missed investment opportunities.

- Business motivation:
  - Improve customer targeting
  - Reduce inefficient marketing spend
  - Provide better property recommendations
  - Improve investor targeting
  - Identify missed investment opportunities
- Background:
  - Real estate buyers have diverse characteristics and motivations
  - Major buyer categories include individual buyers, institutional investors,
    international buyers, high-net-worth investors, and first-time buyers
- Role of AI:
  - Use machine learning clustering techniques to discover hidden patterns
    in buyer behavior
  - Group customers into meaningful buyer segments



Project Initialization

In [1]:
#import libraries

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import sklearn
print(sklearn.__version__)

from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score


1.5.0


Import Data and Create Dataframe

First Dataset : clients.csv
Second Dataset : properties.csv

In [37]:
client = pd.read_csv("C:/Users/Chandan/Desktop/Financial-Analytics-Real-Estate/data/raw/clients.csv")
properties = pd.read_csv("C:/Users/Chandan/Desktop/Financial-Analytics-Real-Estate/data/raw/properties.csv")

In [38]:
client

,client_id,client_type,first_name,last_name,date_of_birth,gender,country,region,acquisition_purpose,satisfaction_score,loan_applied,referral_channel
0,C0001,Individual,Kareem,Liu,05-11-1968,F,USA,California,Home,4,Yes,Website
1,C0002,Individual,Trystan,Oconnor,11/26/1962,M,USA,California,Home,1,No,Website
2,C0003,Individual,Kale,Gay,04-07-1959,M,USA,California,Home,4,Yes,Agency
3,C0004,Individual,Russell,Gross,11/25/1959,M,USA,California,Home,5,No,Website
4,C0005,Company,Marleez,Co,2/28/1976,M,USA,California,Investment,5,No,Website
...,...,...,...,...,...,...,...,...,...,...,...,...
1995,C1996,Individual,Samantha,Brooks,6/20/1992,F,USA,Nevada,Home,1,Yes,Website
1996,C1997,Individual,Matthew,Cox,9/27/1942,M,USA,Arizona,Home,4,Yes,Website
1997,C1998,Individual,Mark,Patterson,9/13/1963,M,USA,Virginia,Home,5,Yes,Website
1998,C1999,Individual,Lawrence,Montgomery,6/15/1944,M,USA,Ohio,Home,4,Yes,Client


In [39]:
properties

,listing_id,tower_number,transaction_date,unit_category,unit_number,floor_area_sqft,sale_price,listing_status,client_ref
0,1012,1,01-01-2024,Apartment,12,1160.36,"$300,385.62",Sold,C0027
1,1015,1,01-01-2024,Apartment,15,782.25,"$208,930.81",Sold,C0097
2,1021,1,01-01-2024,Apartment,21,756.21,"$218,585.92",Sold,C0113
3,1030,1,01-01-2024,Apartment,30,743.09,"$246,172.68",Sold,C0141
4,2016,2,01-01-2024,Apartment,16,701.66,"$212,265.67",Sold,C0146
...,...,...,...,...,...,...,...,...,...
9995,200417,20,12-01-2025,Apartment,48,860.80,"$271,177.34",Available,NaN
9996,200429,20,12-01-2025,Apartment,48,860.80,"$265,186.67",Sold,C1412
9997,200430,20,12-01-2025,Apartment,48,860.80,"$288,189.74",Sold,C0504
9998,200449,20,12-01-2025,Apartment,51,1062.33,"$328,980.19",Available,NaN


In [7]:
client.head(3)

,client_id,client_type,first_name,last_name,date_of_birth,gender,country,region,acquisition_purpose,satisfaction_score,loan_applied,referral_channel
0,C0001,Individual,Kareem,Liu,05-11-1968,F,USA,California,Home,4,Yes,Website
1,C0002,Individual,Trystan,Oconnor,11/26/1962,M,USA,California,Home,1,No,Website
2,C0003,Individual,Kale,Gay,04-07-1959,M,USA,California,Home,4,Yes,Agency


In [29]:
client.sample()

,client_id,client_type,first_name,last_name,date_of_birth,gender,country,region,acquisition_purpose,satisfaction_score,loan_applied,referral_channel
777,C0778,Individual,George,Pierce,1998-07-22,M,USA,California,Investment,4,Yes,Agency


In [9]:
client.shape

(2000, 12)

In [14]:
client.columns

Index(['client_id', 'client_type', 'first_name', 'last_name', 'date_of_birth',
       'gender', 'country', 'region', 'acquisition_purpose',
       'satisfaction_score', 'loan_applied', 'referral_channel'],
      dtype='object')

Data Cleaning

In [8]:
client.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   client_id            2000 non-null   object
 1   client_type          2000 non-null   object
 2   first_name           2000 non-null   object
 3   last_name            2000 non-null   object
 4   date_of_birth        2000 non-null   object
 5   gender               2000 non-null   object
 6   country              2000 non-null   object
 7   region               2000 non-null   object
 8   acquisition_purpose  2000 non-null   object
 9   satisfaction_score   2000 non-null   int64 
 10  loan_applied         2000 non-null   object
 11  referral_channel     2000 non-null   object
dtypes: int64(1), object(11)
memory usage: 187.6+ KB


In [11]:
#Changing the data type if the date_of_birth

client['date_of_birth'] = pd.to_datetime(client['date_of_birth'], format='mixed', errors='coerce')

In [13]:
client.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   client_id            2000 non-null   object        
 1   client_type          2000 non-null   object        
 2   first_name           2000 non-null   object        
 3   last_name            2000 non-null   object        
 4   date_of_birth        2000 non-null   datetime64[ns]
 5   gender               2000 non-null   object        
 6   country              2000 non-null   object        
 7   region               2000 non-null   object        
 8   acquisition_purpose  2000 non-null   object        
 9   satisfaction_score   2000 non-null   int64         
 10  loan_applied         2000 non-null   object        
 11  referral_channel     2000 non-null   object        
dtypes: datetime64[ns](1), int64(1), object(10)
memory usage: 187.6+ KB


In [20]:
client.isnull().sum()

client_id              0
client_type            0
first_name             0
last_name              0
date_of_birth          0
gender                 0
country                0
region                 0
acquisition_purpose    0
satisfaction_score     0
loan_applied           0
referral_channel       0
dtype: int64

In [26]:
client.duplicated(['client_id']).sum()

0

In [30]:
cat_cols = client.columns

for col in cat_cols:
    print(f"\n--- {col} ---")
    print(client[col].value_counts(dropna=False))


--- client_id ---
client_id
C0001    1
C1330    1
C1343    1
C1342    1
C1341    1
        ..
C0662    1
C0661    1
C0660    1
C0659    1
C2000    1
Name: count, Length: 2000, dtype: int64

--- client_type ---
client_type
Individual    1897
Company        103
Name: count, dtype: int64

--- first_name ---
first_name
Carl       24
Gerald     20
Jason      19
Sarah      19
William    19
           ..
Laurel      1
Simon       1
Aniyah      1
Kamden      1
Kareem      1
Name: count, Length: 337, dtype: int64

--- last_name ---
last_name
Co         103
Howard      39
Hansen      34
Morales     34
Cox         33
          ... 
Larsen       1
Stein        1
Perry        1
Mueller      1
Bailey       1
Name: count, Length: 224, dtype: int64

--- date_of_birth ---
date_of_birth
1993-06-14    3
1975-03-27    3
1973-10-24    3
1967-04-13    3
1995-11-12    3
             ..
1948-08-20    1
1977-04-25    1
1965-04-16    1
1964-12-02    1
1944-06-15    1
Name: count, Length: 1889, dtype: int64

--

In [31]:
for col in cat_cols:
    client[col] = (
        client[col]
        .astype(str)                              # avoid errors on NaN/non-strings
        .str.strip()                              # remove leading/trailing spaces
        .str.lower()                              # consistent case
        .str.replace(r"\s+", " ", regex=True)     # collapse multiple spaces
        .str.replace(r"[_\-]+", " ", regex=True)  # underscores/hyphens to space
    )

    # astype(str) turns NaN into the text "nan"; convert it back
    client[col] = client[col].replace({"nan": pd.NA, "none": pd.NA, "": pd.NA,
                               "n/a": pd.NA, "na": pd.NA, "null": pd.NA})

In [32]:
client['gender']

0       f
1       m
2       m
3       m
4       m
       ..
1995    f
1996    m
1997    m
1998    m
1999    f
Name: gender, Length: 2000, dtype: object

In [33]:
for col in cat_cols:
    print(f"\n{col}: {client[col].nunique()} unique values")
    print(sorted(client[col].dropna().unique()))


client_id: 2000 unique values
['c0001', 'c0002', 'c0003', 'c0004', 'c0005', 'c0006', 'c0007', 'c0008', 'c0009', 'c0010', 'c0011', 'c0012', 'c0013', 'c0014', 'c0015', 'c0016', 'c0017', 'c0018', 'c0019', 'c0020', 'c0021', 'c0022', 'c0023', 'c0024', 'c0025', 'c0026', 'c0027', 'c0028', 'c0029', 'c0030', 'c0031', 'c0032', 'c0033', 'c0034', 'c0035', 'c0036', 'c0037', 'c0038', 'c0039', 'c0040', 'c0041', 'c0042', 'c0043', 'c0044', 'c0045', 'c0046', 'c0047', 'c0048', 'c0049', 'c0050', 'c0051', 'c0052', 'c0053', 'c0054', 'c0055', 'c0056', 'c0057', 'c0058', 'c0059', 'c0060', 'c0061', 'c0062', 'c0063', 'c0064', 'c0065', 'c0066', 'c0067', 'c0068', 'c0069', 'c0070', 'c0071', 'c0072', 'c0073', 'c0074', 'c0075', 'c0076', 'c0077', 'c0078', 'c0079', 'c0080', 'c0081', 'c0082', 'c0083', 'c0084', 'c0085', 'c0086', 'c0087', 'c0088', 'c0089', 'c0090', 'c0091', 'c0092', 'c0093', 'c0094', 'c0095', 'c0096', 'c0097', 'c0098', 'c0099', 'c0100', 'c0101', 'c0102', 'c0103', 'c0104', 'c0105', 'c0106', 'c0107', 'c010

In [35]:
client["loan_applied"] = (
    client["loan_applied"]
    .replace({"yes": 1, "no": 0})
    .infer_objects(copy=False)
)

In [36]:
client

,client_id,client_type,first_name,last_name,date_of_birth,gender,country,region,acquisition_purpose,satisfaction_score,loan_applied,referral_channel
0,c0001,individual,kareem,liu,1968 05 11,f,usa,california,home,4,1,website
1,c0002,individual,trystan,oconnor,1962 11 26,m,usa,california,home,1,0,website
2,c0003,individual,kale,gay,1959 04 07,m,usa,california,home,4,1,agency
3,c0004,individual,russell,gross,1959 11 25,m,usa,california,home,5,0,website
4,c0005,company,marleez,co,1976 02 28,m,usa,california,investment,5,0,website
...,...,...,...,...,...,...,...,...,...,...,...,...
1995,c1996,individual,samantha,brooks,1992 06 20,f,usa,nevada,home,1,1,website
1996,c1997,individual,matthew,cox,1942 09 27,m,usa,arizona,home,4,1,website
1997,c1998,individual,mark,patterson,1963 09 13,m,usa,virginia,home,5,1,website
1998,c1999,individual,lawrence,montgomery,1944 06 15,m,usa,ohio,home,4,1,client


Observation:

From the above data cleaning process, it is found that the client dataset  does not any missing values and not any duplicate values.But it is found that the data type of date_of_birth is object instead of data_time. So it is changed to data_time data type.

Also the loan applied is in the form of string("Yes", "No"). It is converted into integer(1 for "Yes" and 0 for "No" ) for better understanding

In [44]:
properties.head(3)

,listing_id,tower_number,transaction_date,unit_category,unit_number,floor_area_sqft,sale_price,listing_status,client_ref
0,1012,1,01-01-2024,Apartment,12,1160.36,"$300,385.62",Sold,C0027
1,1015,1,01-01-2024,Apartment,15,782.25,"$208,930.81",Sold,C0097
2,1021,1,01-01-2024,Apartment,21,756.21,"$218,585.92",Sold,C0113


In [40]:
properties.shape

(10000, 9)

In [42]:
properties.columns

Index(['listing_id', 'tower_number', 'transaction_date', 'unit_category',
       'unit_number', 'floor_area_sqft', 'sale_price', 'listing_status',
       'client_ref'],
      dtype='object')

In [43]:
properties.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   listing_id        10000 non-null  int64  
 1   tower_number      10000 non-null  int64  
 2   transaction_date  10000 non-null  object 
 3   unit_category     10000 non-null  object 
 4   unit_number       10000 non-null  int64  
 5   floor_area_sqft   10000 non-null  float64
 6   sale_price        10000 non-null  object 
 7   listing_status    10000 non-null  object 
 8   client_ref        7305 non-null   object 
dtypes: float64(1), int64(3), object(5)
memory usage: 703.3+ KB


In [45]:
#changing the data type of transaction_date(str -> date_time)
properties['transaction_date'] = pd.to_datetime(properties['transaction_date'])

In [ ]:
#changing the data type of sale_price (str-> float)
properties['sale_price'] = pd.to_numeric(
    properties['sale_price']
    .astype(str)
    .str.replace(r'[$,]', '', regex=True),
    errors='coerce'
)

In [52]:
properties.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   listing_id        10000 non-null  int64         
 1   tower_number      10000 non-null  int64         
 2   transaction_date  10000 non-null  datetime64[ns]
 3   unit_category     10000 non-null  object        
 4   unit_number       10000 non-null  int64         
 5   floor_area_sqft   10000 non-null  float64       
 6   sale_price        10000 non-null  float64       
 7   listing_status    10000 non-null  object        
 8   client_ref        7305 non-null   object        
dtypes: datetime64[ns](1), float64(2), int64(3), object(3)
memory usage: 703.3+ KB


In [54]:
properties.isnull().sum()

listing_id             0
tower_number           0
transaction_date       0
unit_category          0
unit_number            0
floor_area_sqft        0
sale_price             0
listing_status         0
client_ref          2695
dtype: int64

In [ ]:
# to check if  missing client_ref rows correspond to a particular listing status
print(pd.crosstab(properties["listing_status"],
                  properties["client_ref"].isnull()))



client_ref      False  True 
listing_status              
Available           0   2695
Sold             7305      0


In [58]:
#  Share of missing
print(properties["client_ref"].isnull().mean() * 100)

26.950000000000003


In [59]:
# Confirm the rule holds: client_ref is null if and only if the unit is Available
assert ((properties["listing_status"] == "Available") == properties["client_ref"].isnull()).all()

print("Sold units:", (properties["listing_status"] == "Sold").sum())
print("Available units:", (properties["listing_status"] == "Available").sum())

Sold units: 7305
Available units: 2695


In [60]:
# Sanity check numeric columns for impossible values
print(properties[["floor_area_sqft", "sale_price"]].describe())
print((properties["sale_price"] <= 0).sum(), (properties["floor_area_sqft"] <= 0).sum())

       floor_area_sqft     sale_price
count     10000.000000   10000.000000
mean       1139.941412  344374.678683
std         418.373967  131563.099417
min         410.710000   97402.800000
25%         782.200000  232173.642500
50%        1110.880000  330680.845000
75%        1499.000000  448852.575000
max        1957.160000  736652.270000
0 0


In [61]:
# Standardize categorical labels
for col in ["unit_category", "listing_status", "tower_number"]:
    print(properties[col].value_counts(dropna=False))

unit_category
Apartment    8547
Office       1453
Name: count, dtype: int64
listing_status
Sold         7305
Available    2695
Name: count, dtype: int64
tower_number
13    606
18    589
3     574
11    563
2     551
9     537
15    520
7     520
20    502
8     502
5     502
4     485
6     476
14    476
1     476
17    450
19    433
10    432
12    416
16    390
Name: count, dtype: int64


In [56]:
properties.duplicated().sum()

0